In [56]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from datio import get_header, get_forest, get_tree, get_metadata, SIZE_INT, SIZE_DOUBLE

data_dir = r'../../../data'
dat_files = glob.glob(os.path.join(data_dir, '*.dat'))

test_file = dat_files[2]
for dat_file in dat_files:
    print(dat_file)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
../../../data/uni0000.dat
../../../data/td3d.dat
../../../data/weno511_sub_0000.dat
../../../data/tdm.dat


In [3]:
# CORRECTED VERSION: Simple file I/O without mmap (parallel-safe)
from concurrent.futures import ThreadPoolExecutor
def read_blocks_parallel(filename, field_indices=None, max_workers=8):
    """
    Parallel block reading using simple file I/O - no mmap conflicts!
    Each worker opens its own file handle, so no shared state issues.
    """
    
    # Get metadata first
    header, forest, block_info = get_metadata(filename)
    block_offsets = block_info[2]
    block_shape = header['block_nx']
    ndim = header['ndim']
    nw = header['nw']
    
    if field_indices is None:
        field_indices = list(range(nw))
    
    def read_single_block_from_file(offset):
        """Worker function - reads one block using simple file I/O"""
        with open(filename, 'rb') as f:
            f.seek(offset)
            
            # Read ghost cells
            ghostcells = np.frombuffer(f.read(2 * ndim * SIZE_INT), dtype='=i4')
            ghostcells = ghostcells.reshape(2, ndim)
            bg_shape = block_shape + ghostcells[0] + ghostcells[1]
            count = np.prod(bg_shape)
            byte_size_field = count * SIZE_DOUBLE
            
            # Read requested fields
            block_fields = []
            for field_idx in field_indices:
                f.seek(offset + 2 * ndim * SIZE_INT + field_idx * byte_size_field)
                arr = np.frombuffer(f.read(byte_size_field), dtype='=f8')
                arr = arr.reshape(bg_shape[::-1]).T
                
                # Expand to 3D if needed
                while len(arr.shape) < 3:
                    arr = arr[..., np.newaxis]
                
                block_fields.append(arr)
            
            return ghostcells, np.array(block_fields)
    
    # Parallel execution with ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(read_single_block_from_file, block_offsets))
    
    return results


In [4]:
# CORRECTED VERSION: Simple file I/O without mmap (parallel-safe)
from concurrent.futures import ThreadPoolExecutor
import mmap
def read_blocks_mmap_parallel(filename, field_indices=None, max_workers=8):
    """
    Parallel block reading using simple file I/O with mmap.
    Each worker opens its own file handle, so no shared state issues.
    """
    
    # Get metadata first
    header, forest, block_info = get_metadata(filename)
    block_offsets = block_info[2]
    block_shape = header['block_nx']
    ndim = header['ndim']
    nw = header['nw']
    
    if field_indices is None:
        field_indices = list(range(nw))

    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
    
            def read_single_block_from_file(offset):
                """Worker function - reads one block using simple file I/O"""
                    
                    # Read ghost cells
                ghostcells = np.frombuffer(mm, dtype='=i4', count=2*ndim, offset=offset).copy()
                ghostcells = ghostcells.reshape(2, ndim)
                bg_shape = block_shape + ghostcells[0] + ghostcells[1]
                count = np.prod(bg_shape)
                byte_size_field = count * SIZE_DOUBLE
                
                # Read requested fields
                block_fields = []
                for field_idx in field_indices:
                    byte_offset = offset + 2 * ndim * SIZE_INT + field_idx * byte_size_field
                    arr = np.frombuffer(mm, dtype='=f8', count=count, offset=byte_offset).copy()
                    arr = arr.reshape(bg_shape[::-1]).T
                    
                    # Expand to 3D if needed
                    while len(arr.shape) < 3:
                        arr = arr[..., np.newaxis]
                    
                    block_fields.append(arr)
                
                return ghostcells, np.array(block_fields)
            
            # Parallel execution with ThreadPoolExecutor
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                results = list(executor.map(read_single_block_from_file, block_offsets))
    
    return results


# Test the sequential mmap version
print("Testing sequential mmap version...")
results_sequential = read_blocks_mmap_parallel(test_file, field_indices=[0,1,2])
print(f"✓ Success! Read {len(results_sequential)} blocks")
if results_sequential:
    print(f"First block shape: {results_sequential[0][1].shape}")
    print(f"Sample data: {results_sequential[0][1][0,0,0] if results_sequential[0][1] is not None else 'None'}")

Testing sequential mmap version...
✓ Success! Read 23888 blocks
First block shape: (3, 8, 8, 8)
Sample data: [2.0169136  2.01111983 1.99597239 1.98071786 1.96433388 1.94884663
 1.93351261 1.91817654]


In [5]:
# SEQUENTIAL MMAP VERSION: No parallel processing, just mmap for performance
import gc
def read_blocks_mmap_sequential(filename, field_indices=None, max_workers=8):
    """
    Sequential block reading using mmap for performance.
    No parallel processing = no mmap conflicts!
    
    Features:
    - Uses mmap for fast file access
    - Sequential processing (no parallel conflicts)
    - Memory efficient
    - Reliable and simple
    """
    import mmap
    
    # Get metadata first
    header, forest, block_info = get_metadata(filename)
    block_offsets = block_info[2].copy()
    block_shape = header['block_nx'].copy()
    ndim = header['ndim']
    nw = header['nw']
    
    if field_indices is None:
        field_indices = list(range(nw))

    # CRITICAL: Delete all variables that might hold file references
    del header, forest, block_info
    block_offsets = [int(x) for x in block_offsets]
    block_shape = tuple(int(x) for x in block_shape)
    
    print(f"Reading {len(block_offsets)} blocks sequentially with mmap...")
    
    # Open file and create mmap
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            results = []
            
            for i, offset in enumerate(block_offsets):
                ghost_offset = 2 * ndim * SIZE_INT
                
                # Read ghost cells
                ghostcells_view = np.frombuffer(mm, dtype='=i4', count=2*ndim, offset=offset).copy()
                ghostcells = ghostcells_view.reshape(2, ndim).copy()
                del ghostcells_view
                bg_shape = block_shape + ghostcells[0] + ghostcells[1]
                count = np.prod(bg_shape)
                byte_size_field = count * SIZE_DOUBLE
                
                # Read requested fields
                block_fields = []
                for field_idx in field_indices:
                    byte_offset = offset + ghost_offset + field_idx * byte_size_field
                    arr = np.frombuffer(mm, dtype='=f8', count=count, offset=byte_offset).copy()
                    arr = arr.reshape(bg_shape[::-1]).T
                    
                    # Expand to 3D if needed
                    while len(arr.shape) < 3:
                        arr = arr[..., np.newaxis]
                    
                    block_fields.append(arr)  # Copy to avoid mmap dependency
                
                results.append((ghostcells, np.array(block_fields)))
    
    print(f"✓ Completed! Read {len(results)} blocks")
    return results

# Test the sequential mmap version
print("Testing sequential mmap version...")
results_sequential = read_blocks_mmap_sequential(test_file, field_indices=[0,1,2])
print(f"✓ Success! Read {len(results_sequential)} blocks")
if results_sequential:
    print(f"First block shape: {results_sequential[0][1].shape}")
    print(f"Sample data: {results_sequential[0][1][0,0,0] if results_sequential[0][1] is not None else 'None'}")


Testing sequential mmap version...
Reading 23888 blocks sequentially with mmap...
✓ Completed! Read 23888 blocks
✓ Success! Read 23888 blocks
First block shape: (3, 8, 8, 8)
Sample data: [2.0169136  2.01111983 1.99597239 1.98071786 1.96433388 1.94884663
 1.93351261 1.91817654]


In [7]:
def read_blocks_sequential(filename, field_indices=None, max_workers=8):
    """
    Parallel block reading using simple file I/O - no mmap conflicts!
    Each worker opens its own file handle, so no shared state issues.
    """
    
    # Get metadata first
    header, forest, block_info = get_metadata(filename)
    block_offsets = block_info[2]
    block_shape = header['block_nx']
    ndim = header['ndim']
    nw = header['nw']
    
    if field_indices is None:
        field_indices = list(range(nw))
    
    results = []
    with open(filename, 'rb') as f:
        for offset in block_offsets:
            f.seek(offset)
            
            # Read ghost cells
            ghostcells = np.frombuffer(f.read(2 * ndim * SIZE_INT), dtype='=i4')
            ghostcells = ghostcells.reshape(2, ndim)
            bg_shape = block_shape + ghostcells[0] + ghostcells[1]
            count = np.prod(bg_shape)
            byte_size_field = count * SIZE_DOUBLE
            
            # Read requested fields
            block_fields = []
            for field_idx in field_indices:
                f.seek(offset + 2 * ndim * SIZE_INT + field_idx * byte_size_field)
                arr = np.frombuffer(f.read(byte_size_field), dtype='=f8')
                arr = arr.reshape(bg_shape[::-1]).T
                
                # Expand to 3D if needed
                while len(arr.shape) < 3:
                    arr = arr[..., np.newaxis]
                
                block_fields.append(arr)
            
            results.append((ghostcells, np.array(block_fields)))
    
    return results

print("Testing sequential file I/O...")
results_sequential = read_blocks_sequential(test_file, field_indices=[0,1,2])
print(f"✓ Success! Read {len(results_sequential)} blocks")
if results_sequential:
    print(f"First block shape: {results_sequential[0][1].shape}")
    print(f"Sample data: {results_sequential[0][1][0,0,0] if results_sequential[0][1] is not None else 'None'}")


Testing sequential file I/O...
✓ Success! Read 23888 blocks
First block shape: (3, 8, 8, 8)
Sample data: [2.0169136  2.01111983 1.99597239 1.98071786 1.96433388 1.94884663
 1.93351261 1.91817654]


In [22]:
# USAGE EXAMPLES AND COMPARISON
def benchmark_parallel_strategies(filename, block_offsets, field_indices=[0,1,2], max_workers=8):
    """
    Benchmark different parallel reading strategies.
    """
    import time
    
    print("=" * 60)
    print("BENCHMARKING PARALLEL BLOCK READING STRATEGIES")
    print("=" * 60)
    print(f"File: {filename}")
    print(f"Blocks: {len(block_offsets)}")
    print(f"Workers: {max_workers}")
    print(f"Fields: {field_indices}")
    print()
    
    strategies = [
        ("Max Performance (Shared mmap)", read_blocks_mmap_parallel),
        ("Simple (No mmap)", read_blocks_parallel),
        ("Sequential mmap", read_blocks_mmap_sequential),
        ("Sequential simple", read_blocks_sequential),
    ]
    
    results = {}
    
    for name, func in strategies:
        print(f"Testing: {name}")
        try:
            start_time = time.time()
            data = func(filename, field_indices, max_workers)
            end_time = time.time()
            
            duration = end_time - start_time
            results[name] = {
                'duration': duration,
                'success': True,
                'blocks_read': len(data) if data else 0
            }
            
            print(f"✓ Success: {duration:.2f}s, {len(data)} blocks read")
            
        except Exception as e:
            results[name] = {
                'duration': float('inf'),
                'success': False,
                'error': str(e)
            }
            print(f"✗ Failed: {e}")
        
        print()
    
    # Summary
    print("=" * 60)
    print("BENCHMARK SUMMARY")
    print("=" * 60)
    
    successful_results = {k: v for k, v in results.items() if v['success']}
    if successful_results:
        fastest = min(successful_results.items(), key=lambda x: x[1]['duration'])
        print(f"🏆 Fastest: {fastest[0]} ({fastest[1]['duration']:.2f}s)")
        print()
        
        print("All successful strategies:")
        for name, result in sorted(successful_results.items(), key=lambda x: x[1]['duration']):
            print(f"  {name}: {result['duration']:.2f}s")
    
    if any(not v['success'] for v in results.values()):
        print("\nFailed strategies:")
        for name, result in results.items():
            if not result['success']:
                print(f"  {name}: {result['error']}")
    
    return results

# Example usage:
# Get some block offsets to test with
header, forest, block_info = get_metadata(test_file)
test_offsets = block_info[2][:10]  # Test with first 10 blocks

# Run benchmark
benchmark_results = benchmark_parallel_strategies(test_file, test_offsets, [0,1,2], max_workers=8)


BENCHMARKING PARALLEL BLOCK READING STRATEGIES
File: ../../../data/weno511_sub_0000.dat
Blocks: 10
Workers: 8
Fields: [0, 1, 2]

Testing: Max Performance (Shared mmap)
✓ Success: 3.13s, 23888 blocks read

Testing: Simple (No mmap)
✓ Success: 4.12s, 23888 blocks read

Testing: Sequential mmap
Reading 23888 blocks sequentially with mmap...
✓ Completed! Read 23888 blocks
✓ Success: 0.99s, 23888 blocks read

Testing: Sequential simple
✓ Success: 0.87s, 23888 blocks read

BENCHMARK SUMMARY
🏆 Fastest: Sequential simple (0.87s)

All successful strategies:
  Sequential simple: 0.87s
  Sequential mmap: 0.99s
  Max Performance (Shared mmap): 3.13s
  Simple (No mmap): 4.12s


In [34]:
from datio import read_blocks_sequential

# Test the sequential mmap version
print("Testing sequential mmap version...")
results_sequential = read_blocks_sequential(test_file, field_indices=[0,1,2])
print(f"✓ Success! Read {len(results_sequential)} blocks")
print(f"First block shape: {results_sequential[0].shape}")
print("Sample data:", results_sequential[0][0,0,:,0])

Testing sequential mmap version...
✓ Success! Read 23888 blocks
First block shape: (8, 8, 8, 3)
Sample data: [2.0169136  2.01111983 1.99597239 1.98071786 1.96433388 1.94884663
 1.93351261 1.91817654]


In [98]:
from datio import get_metadata
from amrvac_dataset import AMRVACDataSet

ds = AMRVACDataSet(test_file)

ValueError: Buffer dtype mismatch, expected 'int' but got 'double'

In [74]:
ds.metadata

{'datfile_version': 5,
 'offset_tree': 320,
 'offset_blocks': 682832,
 'nw': 7,
 'ndir': 3,
 'ndim': 3,
 'levmax': 6,
 'nleafs': 23888,
 'nparents': 3412,
 'it': 16705,
 'time': 4.4,
 'xmin': array([-1.33333333, -1.33333333,  0.        ]),
 'xmax': array([1.33333333, 1.33333333, 1.33333333]),
 'domain_nx': array([16, 16,  8]),
 'block_nx': array([8, 8, 8]),
 'periodic': array([False, False, False]),
 'geometry': 'Cartesian_3D',
 'staggered': True,
 'w_names': ['rho', 'm1', 'm2', 'm3', 'b1', 'b2', 'b3'],
 'physics_type': 'mhd',
 'n_par': 1,
 'params': array([1.]),
 'param_names': ['gamma'],
 'snapshotnext': 12,
 'slicenext': 0,
 'collapsenext': 0}

In [76]:
ds.uniform_grid([-1, -1, 0.3], [1, 1, 1.3], [100, 100, 100])

AttributeError: 'AMRVACDataSet' object has no attribute 'mesh'

In [36]:
header, forest, block_info = get_metadata(test_file)